# Assignment 8 — Model Monitoring using Prometheus and Grafana

## Objective

The objective of this assignment is to implement model monitoring using
**Prometheus** and **Grafana**.

A machine learning model will be trained using the **Wine Quality dataset**
and integrated with a prediction service. Custom application and model
metrics will be exposed in a Prometheus-compatible format, collected by
Prometheus, and visualized through Grafana.

The monitoring workflow implemented in this assignment is:

**Wine Quality Dataset → Machine Learning Model → Prediction Service → Prometheus → Grafana**

### Key Components

- **Dataset:** Wine Quality
- **Machine Learning Model:** Classification model
- **Model Serving:** FastAPI
- **Metrics Collection:** Prometheus Client
- **Monitoring:** Prometheus
- **Visualization:** Grafana

### Metrics to be Monitored

The application will expose metrics such as:

- Total prediction requests
- Successful predictions
- Prediction errors
- Prediction latency
- Prediction confidence
- Prediction distribution

This assignment demonstrates how machine learning applications can be
monitored using standard observability tools rather than only monitoring
the model's predictive performance during training.

## 1. Import Libraries and Load Dataset

The required Python libraries are imported for dataset handling and file management.

The notebook also checks whether the Wine Quality dataset is already available in the `data/` directory. If the dataset is not found, it is downloaded automatically, extracted, and then loaded into a pandas DataFrame. 

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

DATASET_PATH = DATA_DIR / "winequality-red.csv"
ZIP_PATH = DATA_DIR / "winequality.zip"

DATA_URL = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"

# Download the dataset only if it is not already available
if DATASET_PATH.exists():
    print(f"Dataset found: {DATASET_PATH}")

else:
    print("Dataset not found. Downloading...")

    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

    print("Download completed.")

    # Extract the dataset
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

    # Remove the ZIP file after extraction
    ZIP_PATH.unlink()

    print(f"Dataset extracted to: {DATA_DIR}")

# Load the dataset
df = pd.read_csv(DATASET_PATH, sep=";")

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")

Dataset not found. Downloading...
Download completed.
Dataset extracted to: d:\College\3rd_Year\MLOPS\Assignments\Assignment-8-Prometheus-and-Grafana\data
Dataset loaded successfully.
Shape: (1599, 12)


## 2. Dataset Overview

The Wine Quality dataset contains physicochemical measurements of red wine samples along with a quality score.

The dataset contains numerical features related to the chemical properties of wine. The `quality` column represents the quality score assigned to each sample.

Before training the machine learning model, the dataset will be inspected to understand its structure, feature types, and basic statistical properties.

In [2]:
# Display the first five rows
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## 3. Dataset Structure and Data Types

The structure of the dataset is examined to identify the number of observations,
features, and their corresponding data types.

Since the Wine Quality dataset is expected to contain numerical features,
checking the data types also helps verify that the dataset has been loaded
correctly and is suitable for numerical machine learning algorithms.

In [3]:
# Display dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


## 4. Statistical Summary

Descriptive statistics are calculated for all numerical columns to examine
the central tendency, spread, and range of the dataset features.

This provides an initial understanding of the distribution and scale of the
variables before the data is prepared for model training.

In [4]:
# Display descriptive statistics
df.describe().T

,count,mean,std,min,25%,50%,75%,max
fixed acidity,1599.0,8.319637,1.741096,4.60000,7.1000,7.90000,9.200000,15.90000
volatile acidity,1599.0,0.527821,0.179060,0.12000,0.3900,0.52000,0.640000,1.58000
citric acid,1599.0,0.270976,0.194801,0.00000,0.0900,0.26000,0.420000,1.00000
residual sugar,1599.0,2.538806,1.409928,0.90000,1.9000,2.20000,2.600000,15.50000
chlorides,1599.0,0.087467,0.047065,0.01200,0.0700,0.07900,0.090000,0.61100
free sulfur dioxide,1599.0,15.874922,10.460157,1.00000,7.0000,14.00000,21.000000,72.00000
total sulfur dioxide,1599.0,46.467792,32.895324,6.00000,22.0000,38.00000,62.000000,289.00000
density,1599.0,0.996747,0.001887,0.99007,0.9956,0.99675,0.997835,1.00369
pH,1599.0,3.311113,0.154386,2.74000,3.2100,3.31000,3.400000,4.01000
sulphates,1599.0,0.658149,0.169507,0.33000,0.5500,0.62000,0.730000,2.00000


## 5. Missing Value Analysis

The dataset is checked for missing values in each column.

Identifying missing values before preprocessing is important because machine
learning models generally require complete numerical input data. The result
will determine whether any missing-value handling is required before model
training.

In [5]:
# Check for missing values
missing_values = df.isnull().sum()

missing_values

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

## 6. Target Variable Distribution

The distribution of the original `quality` scores is examined to understand
how the wine samples are distributed across the available quality levels.

This analysis will help determine an appropriate transformation of the
original quality score into the binary classification target used by the
machine learning model.

In [6]:
# Display the distribution of quality scores
quality_distribution = df["quality"].value_counts().sort_index()

quality_distribution

quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64

## 7. Create Binary Classification Target

The original `quality` variable contains quality scores ranging from 3 to 8.
For this assignment, the problem will be transformed into a binary
classification task.

The new target variable will classify wine samples as:

- `0` — Bad: quality score below 6
- `1` — Good: quality score of 6 or above

The original `quality` column will be retained in the dataset for reference,
while `quality_label` will be used as the machine learning target.

In [7]:
# Create binary classification target
df["quality_label"] = (df["quality"] >= 6).astype(int)

# Display the new target distribution
df["quality_label"].value_counts().sort_index()

quality_label
0    744
1    855
Name: count, dtype: int64

## 8. Prepare Features and Target

The dataset is divided into input features and the target variable.

The original `quality` column is excluded from the input features because
`quality_label` is derived directly from it. Including `quality` as a model
feature would introduce target leakage and would not represent a realistic
prediction scenario.

The remaining 11 physicochemical measurements will be used as input features,
while `quality_label` will be used as the target.

In [8]:
# Separate input features and target
feature_columns = [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
]

X = df[feature_columns]
y = df["quality_label"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Number of features: {X.shape[1]}")

Features shape: (1599, 11)
Target shape: (1599,)
Number of features: 11


## 9. Train-Test Split

The prepared dataset is divided into training and testing subsets.

A stratified split is used to preserve the distribution of the binary target
variable in both subsets. A fixed random state is also used to make the
experiment reproducible.

The dataset will be divided into:

- 80% training data
- 20% testing data

In [9]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training features: {X_train.shape}")
print(f"Testing features: {X_test.shape}")
print(f"Training target: {y_train.shape}")
print(f"Testing target: {y_test.shape}")

Training features: (1279, 11)
Testing features: (320, 11)
Training target: (1279,)
Testing target: (320,)


## 10. Train the Machine Learning Model

A Random Forest Classifier is used to learn the relationship between the
physicochemical properties of wine and the binary quality label.

Random Forest is selected as the classification model for this assignment
because it can handle numerical features effectively and provides class
probabilities, which will later be used to expose prediction confidence
through the monitoring system.

The same trained model will be saved and used by the prediction service.

In [10]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest model
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


## 11. Evaluate Model Performance

The trained Random Forest model is evaluated on the test set using standard
classification metrics.

Accuracy, precision, recall, F1-score, and ROC-AUC are calculated to provide
a baseline assessment of the model before it is integrated with the
monitoring system.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Generate predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.8156
Precision: 0.8373
Recall   : 0.8129
F1-Score : 0.8249
ROC-AUC  : 0.9057

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.82      0.81       149
           1       0.84      0.81      0.82       171

    accuracy                           0.82       320
   macro avg       0.81      0.82      0.82       320
weighted avg       0.82      0.82      0.82       320



## 12. Feature Importance

The Random Forest model provides an importance value for each input feature.
These values indicate the relative contribution of the physicochemical
features to the model's predictions.

The feature importance values are inspected to understand which variables
the trained model relied on most strongly.

In [12]:
# Create a feature-importance table
feature_importance = (
    pd.DataFrame({
        "feature": feature_columns,
        "importance": model.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance

,feature,importance
0,alcohol,0.178138
1,sulphates,0.138765
2,volatile acidity,0.113982
3,total sulfur dioxide,0.098226
4,density,0.090872
5,chlorides,0.070574
6,fixed acidity,0.068242
7,pH,0.066011
8,citric acid,0.061240
9,free sulfur dioxide,0.058907


## 13. Save the Trained Model

The trained Random Forest model is saved to the project's `model` directory
so that it can be loaded later by the prediction service.

Saving the trained model separately from the training notebook allows the
monitoring application to use the same trained model without repeating the
training process.

In [13]:
import joblib

# Create the model directory if it does not exist
MODEL_DIR = PROJECT_ROOT / "model"
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "wine_quality_model.pkl"

# Save the trained model
joblib.dump(model, MODEL_PATH)

print(f"Model saved successfully to: {MODEL_PATH}")

Model saved successfully to: d:\College\3rd_Year\MLOPS\Assignments\Assignment-8-Prometheus-and-Grafana\model\wine_quality_model.pkl


## 14. Verify Saved Model

The saved model is loaded from disk to verify that the serialized model can
be successfully restored.

This confirms that the model artifact generated during training is ready to
be consumed by the prediction service.

In [14]:
# Load the saved model
loaded_model = joblib.load(MODEL_PATH)

print("Saved model loaded successfully.")
print(f"Model type: {type(loaded_model).__name__}")

Saved model loaded successfully.
Model type: RandomForestClassifier


## 15. Training Summary

The Wine Quality dataset was prepared for binary classification by converting
the original quality scores into two classes.

The Random Forest Classifier was trained using the 11 physicochemical
features and evaluated on a stratified test set.

The trained model achieved the following baseline results:

- Accuracy: 0.8156
- Precision: 0.8373
- Recall: 0.8129
- F1-Score: 0.8249
- ROC-AUC: 0.9057

The trained model has been saved as `wine_quality_model.pkl` in the `model`
directory and successfully loaded again to verify the model artifact.

This model will now be integrated into a FastAPI prediction service, where
Prometheus metrics will be exposed for monitoring through Grafana.